# Clipt Detection Models — Training Notebook v2
#
# INSTRUCTIONS:
# 1. Set runtime to A100 GPU (Runtime → Change runtime type)
# 2. Add ROBOFLOW_API_KEY to Colab Secrets (key icon in sidebar)
# 3. Run chunks IN ORDER — 1, 2, 3, 4
# 4. After each chunk's DOWNLOAD cell runs, save the files
#    before starting the next chunk
# 5. Total training time: ~2.5 hours on A100, ~5 hours on T4
#
# If Colab disconnects mid-session, you never lose more than
# one chunk of work. Re-run from the last incomplete chunk.
#
# Model Filename Cross-Reference (MUST match code exactly):
# ```
# CHUNK 1 — Basketball
#   basketball_jersey_number_v2.pt
#   basketball_jersey_number_v3.pt
#   basketball_player_detector_v2.pt
#
# CHUNK 2 — Football
#   football_positions_detector.pt
#   football_presnap_detector.pt
#   jersey_number_universal_v1.pt
#   jersey_number_universal_v2.pt
#
# CHUNK 3 — Lacrosse
#   lacrosse_detector_v1.pt
#   lacrosse_detector_v2.pt
#
# CHUNK 4 — Ball / Zone / Action
#   basketball_ball_detector.pt
#   football_ball_detector.pt
#   basketball_court_zones.pt
#   basketball_action_detector.pt
#   lacrosse_ball_detector.pt
# ```

## Setup — Run this first (every session)

In [ ]:
import os

# ── GPU verification ──────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "GPU not active — go to Runtime → Change runtime type → A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Read API key from Colab Secrets (key icon in sidebar) ─────
from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')

!pip install roboflow ultralytics -q
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key=api_key)
print("Setup complete — GPU active, Roboflow authenticated")

# ════════════════════════════════════════════════════
# CHUNK 1 — Critical Basketball Models (~45 min)
# ════════════════════════════════════════════════════
# Train these first — they fix the broken basketball jersey OCR
# After this chunk finishes, download all 3 files before continuing

In [ ]:
# ── Cell 1A — Download basketball datasets ──────────────────

def safe_download(workspace, project_slug, version, fallback_version=1, fmt="yolov8"):
    """Download dataset with error handling + auto-retry on version-1."""
    try:
        proj = rf.workspace(workspace).project(project_slug)
        ds = proj.version(version).download(fmt)
        print(f"  Downloaded: {ds.location}")
        return ds
    except Exception as e:
        print(f"  Version {version} failed: {e}")
        if version != fallback_version:
            print(f"  Retrying with version {fallback_version}...")
            try:
                ds = proj.version(fallback_version).download(fmt)
                print(f"  Downloaded (fallback): {ds.location}")
                return ds
            except Exception as e2:
                print(f"  Fallback also failed: {e2}")
        print(f"  SKIPPED — download failed completely")
        return None

print("Downloading basketball datasets...")

print("\n1/3 Basketball jersey v2 (6,932 images):")
dataset_bball_v2 = safe_download("volleyai-actions", "jersey-number-detection-s01j4", 2)

print("\n2/3 Basketball jersey v3 (3,615 images):")
dataset_bball_v3 = safe_download("roboflow-jvuqo", "basketball-jersey-numbers-ocr", 5)

print("\n3/3 Basketball players (1,398 images):")
dataset_bball_players = safe_download("roboflow-universe-projects", "basketball-players-fy4c2", 1)

print("\n✅ Basketball datasets ready" if all([dataset_bball_v2, dataset_bball_v3, dataset_bball_players]) else "\n⚠️ Some basketball datasets failed — check above")

In [ ]:
# ── Cell 1B — Train basketball_jersey_number_v2 ─────────────
# 6,932 images — large dataset settings
if dataset_bball_v2 is not None:
    print("=" * 60)
    print("TRAINING: basketball_jersey_number_v2 (6,932 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_bball_v2.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        name="basketball_jersey_number_v2",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/basketball_jersey_number_v2/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"basketball_jersey_number_v2 mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: basketball_jersey_number_v2 — dataset not available")

In [ ]:
# ── Cell 1C — Train basketball_jersey_number_v3 ─────────────
# 3,615 images — large dataset settings
if dataset_bball_v3 is not None:
    print("=" * 60)
    print("TRAINING: basketball_jersey_number_v3 (3,615 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_bball_v3.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        name="basketball_jersey_number_v3",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/basketball_jersey_number_v3/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"basketball_jersey_number_v3 mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: basketball_jersey_number_v3 — dataset not available")

In [ ]:
# ── Cell 1D — Train basketball_player_detector_v2 ──────────
# 1,398 images — large dataset settings
if dataset_bball_players is not None:
    print("=" * 60)
    print("TRAINING: basketball_player_detector_v2 (1,398 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_bball_players.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        name="basketball_player_detector_v2",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/basketball_player_detector_v2/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"basketball_player_detector_v2 mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: basketball_player_detector_v2 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════
# CHUNK 1 DOWNLOAD — Run this before Chunk 2
# ════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os
from ultralytics import YOLO

chunk1_models = {
    "basketball_jersey_number_v2.pt":
        "runs/detect/basketball_jersey_number_v2/weights/best.pt",
    "basketball_jersey_number_v3.pt":
        "runs/detect/basketball_jersey_number_v3/weights/best.pt",
    "basketball_player_detector_v2.pt":
        "runs/detect/basketball_player_detector_v2/weights/best.pt",
}

passed = []
failed = []
for name, path in chunk1_models.items():
    if os.path.exists(path):
        m = YOLO(path)
        metrics = m.val()
        map50 = metrics.box.map50
        if map50 >= 0.5:
            shutil.copy(path, name)
            size_mb = os.path.getsize(name) / 1024 / 1024
            files.download(name)
            passed.append(f"✅ {name} — mAP50: {map50:.3f}, {size_mb:.1f}MB")
        else:
            failed.append(f"❌ {name} — mAP50: {map50:.3f} (too low)")
    else:
        failed.append(f"❌ {name} — MISSING (training may have failed)")

print("\nPASSED:")
for m in passed: print(f"  {m}")
print("\nFAILED:")
for m in (failed or ["  (none)"]): print(f"  {m}")
print(f"\nCHUNK 1 COMPLETE — save these {len(passed)} files before starting Chunk 2")

# ════════════════════════════════════════════════════
# CHUNK 2 — Football Models (~45 min)
# ════════════════════════════════════════════════════
# Improves football jersey detection, adds position detection
# Download all files after this chunk before continuing

In [ ]:
# ── Cell 2A — Download football datasets ─────────────────

print("Downloading football datasets...")

print("\n1/4 Football positions (755 images):")
dataset_fb_positions = safe_download("bronkscottema", "football-players-zm06l", 15)

print("\n2/4 Football presnap (828 images):")
dataset_fb_presnap = safe_download("football-tracking", "football-presnap-tracker", 1)

print("\n3/4 Universal jersey v1 (826 images):")
dataset_universal_v1 = safe_download("dark-blue-jt0mg", "jerseynumbers", 5)

print("\n4/4 Universal jersey v2 (556 images):")
dataset_universal_v2 = safe_download("yakovk", "jersey-numbers-i1wn5", 1)

print("\n✅ Football datasets ready" if all([dataset_fb_positions, dataset_fb_presnap, dataset_universal_v1, dataset_universal_v2]) else "\n⚠️ Some football datasets failed — check above")

In [ ]:
# ── Cell 2B — Train football_positions_detector ────────────
# 755 images — medium dataset settings
if dataset_fb_positions is not None:
    print("=" * 60)
    print("TRAINING: football_positions_detector (755 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_fb_positions.location}/data.yaml",
        epochs=80,
        imgsz=640,
        batch=16,
        name="football_positions_detector",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/football_positions_detector/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"football_positions_detector mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: football_positions_detector — dataset not available")

In [ ]:
# ── Cell 2C — Train football_presnap_detector ──────────────
# 828 images — medium dataset settings
if dataset_fb_presnap is not None:
    print("=" * 60)
    print("TRAINING: football_presnap_detector (828 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_fb_presnap.location}/data.yaml",
        epochs=80,
        imgsz=640,
        batch=16,
        name="football_presnap_detector",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/football_presnap_detector/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"football_presnap_detector mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: football_presnap_detector — dataset not available")

In [ ]:
# ── Cell 2D — Train jersey_number_universal_v1 ─────────────
# 826 images — medium dataset settings
if dataset_universal_v1 is not None:
    print("=" * 60)
    print("TRAINING: jersey_number_universal_v1 (826 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_universal_v1.location}/data.yaml",
        epochs=80,
        imgsz=640,
        batch=16,
        name="jersey_number_universal_v1",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/jersey_number_universal_v1/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"jersey_number_universal_v1 mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: jersey_number_universal_v1 — dataset not available")

In [ ]:
# ── Cell 2E — Train jersey_number_universal_v2 ─────────────
# 556 images — medium dataset settings
if dataset_universal_v2 is not None:
    print("=" * 60)
    print("TRAINING: jersey_number_universal_v2 (556 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_universal_v2.location}/data.yaml",
        epochs=80,
        imgsz=640,
        batch=16,
        name="jersey_number_universal_v2",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/jersey_number_universal_v2/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"jersey_number_universal_v2 mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: jersey_number_universal_v2 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════
# CHUNK 2 DOWNLOAD — Run before Chunk 3
# ════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os
from ultralytics import YOLO

chunk2_models = {
    "football_positions_detector.pt":
        "runs/detect/football_positions_detector/weights/best.pt",
    "football_presnap_detector.pt":
        "runs/detect/football_presnap_detector/weights/best.pt",
    "jersey_number_universal_v1.pt":
        "runs/detect/jersey_number_universal_v1/weights/best.pt",
    "jersey_number_universal_v2.pt":
        "runs/detect/jersey_number_universal_v2/weights/best.pt",
}

passed = []
failed = []
for name, path in chunk2_models.items():
    if os.path.exists(path):
        m = YOLO(path)
        metrics = m.val()
        map50 = metrics.box.map50
        if map50 >= 0.5:
            shutil.copy(path, name)
            size_mb = os.path.getsize(name) / 1024 / 1024
            files.download(name)
            passed.append(f"✅ {name} — mAP50: {map50:.3f}, {size_mb:.1f}MB")
        else:
            failed.append(f"❌ {name} — mAP50: {map50:.3f} (too low)")
    else:
        failed.append(f"❌ {name} — MISSING (training may have failed)")

print("\nPASSED:")
for m in passed: print(f"  {m}")
print("\nFAILED:")
for m in (failed or ["  (none)"]): print(f"  {m}")
print(f"\nCHUNK 2 COMPLETE — save these {len(passed)} files before starting Chunk 3")

# ════════════════════════════════════════════════════
# CHUNK 3 — Lacrosse Models (~30 min)
# ════════════════════════════════════════════════════
# Adds lacrosse player, ball, and action detection
# Download all files after this chunk before continuing

In [ ]:
# ── Cell 3A — Download lacrosse datasets ─────────────────

print("Downloading lacrosse datasets...")

print("\n1/2 Lacrosse v1 (528 images):")
dataset_lax_v1 = safe_download("ryseai", "lacrosse-object-detection", 1)

print("\n2/2 Lacrosse v2 (380 images):")
dataset_lax_v2 = safe_download("computer-vision-ho8xk", "sports-computer-vision", 1)

print("\n✅ Lacrosse datasets ready" if all([dataset_lax_v1, dataset_lax_v2]) else "\n⚠️ Some lacrosse datasets failed — check above")

In [ ]:
# ── Cell 3B — Train lacrosse_detector_v1 ─────────────────
# 528 images — medium dataset settings
if dataset_lax_v1 is not None:
    print("=" * 60)
    print("TRAINING: lacrosse_detector_v1 (528 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_lax_v1.location}/data.yaml",
        epochs=80,
        imgsz=640,
        batch=16,
        name="lacrosse_detector_v1",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/lacrosse_detector_v1/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"lacrosse_detector_v1 mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: lacrosse_detector_v1 — dataset not available")

In [ ]:
# ── Cell 3C — Train lacrosse_detector_v2 ─────────────────
# 380 images — small dataset settings
if dataset_lax_v2 is not None:
    print("=" * 60)
    print("TRAINING: lacrosse_detector_v2 (380 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_lax_v2.location}/data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        name="lacrosse_detector_v2",
        patience=20,
        device=0,
        augment=True,
        hsv_s=0.9,
        degrees=10.0,
        scale=0.6,
    )
    metrics = YOLO("runs/detect/lacrosse_detector_v2/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"lacrosse_detector_v2 mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: lacrosse_detector_v2 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════
# CHUNK 3 DOWNLOAD — Run before Chunk 4
# ════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os
from ultralytics import YOLO

chunk3_models = {
    "lacrosse_detector_v1.pt":
        "runs/detect/lacrosse_detector_v1/weights/best.pt",
    "lacrosse_detector_v2.pt":
        "runs/detect/lacrosse_detector_v2/weights/best.pt",
}

passed = []
failed = []
for name, path in chunk3_models.items():
    if os.path.exists(path):
        m = YOLO(path)
        metrics = m.val()
        map50 = metrics.box.map50
        if map50 >= 0.5:
            shutil.copy(path, name)
            size_mb = os.path.getsize(name) / 1024 / 1024
            files.download(name)
            passed.append(f"✅ {name} — mAP50: {map50:.3f}, {size_mb:.1f}MB")
        else:
            failed.append(f"❌ {name} — mAP50: {map50:.3f} (too low)")
    else:
        failed.append(f"❌ {name} — MISSING (training may have failed)")

print("\nPASSED:")
for m in passed: print(f"  {m}")
print("\nFAILED:")
for m in (failed or ["  (none)"]): print(f"  {m}")
print(f"\nCHUNK 3 COMPLETE — save these {len(passed)} files before starting Chunk 4")

# ════════════════════════════════════════════════════
# CHUNK 4 — Ball + Zone + Action Models (~45 min)
# ════════════════════════════════════════════════════
# These power the stat generation pipeline
# Train all, download passing models

In [ ]:
# ── Cell 4A — Download ball/zone/action datasets ──────────

print("Downloading ball/zone/action datasets...")

# Basketball ball + rim detection (6,270 images)
print("\n1/5 Basketball ball/rim (6,270 images):")
dataset_bball_ball = safe_download("basketball-hoop-tsdku", "basketball-and-rim", 1)

# Football ball detection (1,237 images)
# NOTE: This dataset is soccer/football (round ball). No large American
# football ball dataset exists on Roboflow. Soccer ball detection still
# provides useful ball-tracking signal for broadcast footage.
print("\n2/5 Football ball (1,237 images):")
dataset_fb_ball = safe_download("roboflow-jvuqo", "football-ball-detection-rejhg", 1)

# Lacrosse ball — reuse lacrosse v1 dataset (528 images, has 'sports ball' class)
print("\n3/5 Lacrosse ball: reusing dataset_lax_v1 from Chunk 3")
if dataset_lax_v1 is None:
    print("  ⚠️ dataset_lax_v1 not available — re-downloading...")
    dataset_lax_v1 = safe_download("ryseai", "lacrosse-object-detection", 1)

# Basketball court zone segmentation (1,294 images)
# Classes: Center-circle, Paint, basketball-court, three-point-line
print("\n4/5 Basketball court zones (1,294 images):")
dataset_court = safe_download("zy-vevvi", "court-segmentation", 4)
if dataset_court is None:
    print("  Trying fallback: samet-mmrat/basketball-court-detection-2-axedc...")
    dataset_court = safe_download("samet-mmrat", "basketball-court-detection-2-axedc", 1)

# Basketball action detection (1,398 images, 7 classes)
# Classes: player-dribble, player-fall, player-jump-shot, player-layup,
#          player-screen, player-shot-block, rim
print("\n5/5 Basketball actions (1,398 images):")
dataset_action = safe_download("roboflow-jvuqo", "basketball-player-detection-2", 1)
if dataset_action is None:
    print("  Trying fallback: basketball-player-detection-3 v6...")
    try:
        project = rf.workspace("roboflow-jvuqo").project("basketball-player-detection-3-ycjdo")
        dataset_action = project.version(6).download("yolov8")
        print(f"  Downloaded (fallback v6): {dataset_action.location}")
    except Exception as e:
        print(f"  Fallback also failed: {e}")
        dataset_action = None

ready = sum(1 for d in [dataset_bball_ball, dataset_fb_ball, dataset_lax_v1, dataset_court, dataset_action] if d is not None)
print(f"\n{'✅' if ready == 5 else '⚠️'} {ready}/5 ball/zone/action datasets ready")

In [ ]:
# ── Cell 4B — Train basketball_ball_detector ──────────────
# 6,270 images — large dataset settings
if dataset_bball_ball is not None:
    print("=" * 60)
    print("TRAINING: basketball_ball_detector (6,270 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_bball_ball.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        name="basketball_ball_detector",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/basketball_ball_detector/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"basketball_ball_detector mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: basketball_ball_detector — dataset not available")

In [ ]:
# ── Cell 4C — Train football_ball_detector ────────────────
# 1,237 images — large dataset settings
if dataset_fb_ball is not None:
    print("=" * 60)
    print("TRAINING: football_ball_detector (1,237 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_fb_ball.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        name="football_ball_detector",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/football_ball_detector/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"football_ball_detector mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: football_ball_detector — dataset not available")

In [ ]:
# ── Cell 4D — Train basketball_court_zones ────────────────
# 1,294 images — large dataset settings
# Classes: Center-circle, Paint, basketball-court, three-point-line
if dataset_court is not None:
    print("=" * 60)
    print("TRAINING: basketball_court_zones (1,294 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_court.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        name="basketball_court_zones",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/basketball_court_zones/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"basketball_court_zones mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: basketball_court_zones — dataset not available")

In [ ]:
# ── Cell 4E — Train basketball_action_detector ─────────────
# 1,398 images — large dataset settings
# Classes: player-dribble, player-fall, player-jump-shot, player-layup,
#          player-screen, player-shot-block, rim
if dataset_action is not None:
    print("=" * 60)
    print("TRAINING: basketball_action_detector (1,398 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_action.location}/data.yaml",
        epochs=60,
        imgsz=640,
        batch=16,
        name="basketball_action_detector",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/basketball_action_detector/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"basketball_action_detector mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: basketball_action_detector — dataset not available")

In [ ]:
# ── Cell 4F — Train lacrosse_ball_detector ────────────────
# Reuses lacrosse v1 dataset (528 images, has 'sports ball' class)
# 528 images — medium dataset settings
if dataset_lax_v1 is not None:
    print("=" * 60)
    print("TRAINING: lacrosse_ball_detector (528 images)")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=f"{dataset_lax_v1.location}/data.yaml",
        epochs=80,
        imgsz=640,
        batch=16,
        name="lacrosse_ball_detector",
        patience=15,
        device=0,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
    )
    metrics = YOLO("runs/detect/lacrosse_ball_detector/weights/best.pt").val()
    map50 = metrics.box.map50
    print(f"lacrosse_ball_detector mAP50: {map50:.3f}")
    if map50 >= 0.5:
        print("✅ PASS")
    else:
        print("❌ FAIL — will be skipped in download")
else:
    print("SKIPPED: lacrosse_ball_detector — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════
# CHUNK 4 DOWNLOAD — Final chunk
# ════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os
from ultralytics import YOLO

chunk4_models = {
    "basketball_ball_detector.pt":
        "runs/detect/basketball_ball_detector/weights/best.pt",
    "football_ball_detector.pt":
        "runs/detect/football_ball_detector/weights/best.pt",
    "lacrosse_ball_detector.pt":
        "runs/detect/lacrosse_ball_detector/weights/best.pt",
    "basketball_court_zones.pt":
        "runs/detect/basketball_court_zones/weights/best.pt",
    "basketball_action_detector.pt":
        "runs/detect/basketball_action_detector/weights/best.pt",
}

passed = []
failed = []
for name, path in chunk4_models.items():
    if os.path.exists(path):
        m = YOLO(path)
        metrics = m.val()
        map50 = metrics.box.map50
        if map50 >= 0.5:
            shutil.copy(path, name)
            size_mb = os.path.getsize(name) / 1024 / 1024
            files.download(name)
            passed.append(f"✅ {name} — mAP50: {map50:.3f}, {size_mb:.1f}MB")
        else:
            failed.append(f"❌ {name} — mAP50: {map50:.3f} (too low)")
    else:
        failed.append(f"❌ {name} — MISSING (training may have failed)")

print("\nPASSED:")
for m in passed: print(f"  {m}")
print("\nFAILED:")
for m in (failed or ["  (none)"]): print(f"  {m}")
print(f"\nCHUNK 4 COMPLETE — all {len(passed)} passing models downloaded")

# ════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════

In [ ]:
import os
from ultralytics import YOLO

print("=" * 60)
print("TRAINING COMPLETE — FINAL REPORT")
print("=" * 60)

all_models = {
    # Chunk 1 — Basketball
    "basketball_jersey_number_v2.pt": "Chunk 1",
    "basketball_jersey_number_v3.pt": "Chunk 1",
    "basketball_player_detector_v2.pt": "Chunk 1",
    # Chunk 2 — Football
    "football_positions_detector.pt": "Chunk 2",
    "football_presnap_detector.pt": "Chunk 2",
    "jersey_number_universal_v1.pt": "Chunk 2",
    "jersey_number_universal_v2.pt": "Chunk 2",
    # Chunk 3 — Lacrosse
    "lacrosse_detector_v1.pt": "Chunk 3",
    "lacrosse_detector_v2.pt": "Chunk 3",
    # Chunk 4 — Ball / Zone / Action
    "basketball_ball_detector.pt": "Chunk 4",
    "football_ball_detector.pt": "Chunk 4",
    "lacrosse_ball_detector.pt": "Chunk 4",
    "basketball_court_zones.pt": "Chunk 4",
    "basketball_action_detector.pt": "Chunk 4",
}

passed = []
failed = []
for name, chunk in all_models.items():
    if os.path.exists(name):
        size_mb = os.path.getsize(name) / 1024 / 1024
        # Validate mAP50
        try:
            m = YOLO(name)
            metrics = m.val()
            map50 = metrics.box.map50
            passed.append(f"✅ {name} ({size_mb:.1f}MB, mAP50: {map50:.3f}) — {chunk}")
        except Exception:
            passed.append(f"✅ {name} ({size_mb:.1f}MB, mAP50: unknown) — {chunk}")
    else:
        # Check if it exists in runs/ but wasn't downloaded (mAP50 too low)
        run_path = f"runs/detect/{name.replace('.pt', '')}/weights/best.pt"
        if os.path.exists(run_path):
            try:
                m = YOLO(run_path)
                metrics = m.val()
                failed.append(f"❌ {name} — mAP50: {metrics.box.map50:.3f} (below 0.5 threshold) — {chunk}")
            except Exception:
                failed.append(f"❌ {name} — trained but validation failed — {chunk}")
        else:
            failed.append(f"❌ {name} — MISSING (training failed or skipped) — {chunk}")

print(f"\n✅ PASSED ({len(passed)}/14):")
for m in passed:
    print(f"  {m}")

print(f"\n❌ FAILED ({len(failed)}/14):")
for m in (failed or ["  (none)"]):
    print(f"  {m}")

print("\n" + "=" * 60)
print("GIT COMMANDS — run after moving .pt files to app/model/:")
print("=" * 60)
print("cd playerJerseyIdentification-master")
print("git add app/model/*.pt")
print('git commit -m "Add v2/v3 Roboflow trained models"')
print("git push")
print("\nRailway will auto-deploy. Check health after ~3 min:")
print("curl https://jersey-detection-production-d8d8.up.railway.app/health")